In [1]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm


In [ ]:
# known issue with statsmodels means there is an issue with the following package until it is resolve
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

### set up paths

In [2]:
#add root path to system path
current_path = os.path.dirname(os.path.realpath(__name__))
root_path = os.path.join(current_path, "../")
sys.path.append(root_path)

In [3]:
# set local paths
stim_path = root_path+"/stimuli/"
# data_path = current_path+"/data/"
print(f"stim_path: {stim_path}")
# print(f"data_path: {data_path}")
os.listdir(stim_path)

stim_path: /workspaces/2025_value_entanglement/anna/..//stimuli/


['grammar_correction_subset48.csv',
 'scenarios_moral_seong_negative.csv',
 'dillion_grammaticality.csv',
 'dillion_2023.csv',
 'scenarios_moral_seong_positive.csv',
 'sentences_grammaticality_helsinki.csv',
 'seong_68.csv',
 'adjectives.json']

### load stimuli from main folder

In [4]:
def return_stimuli_seong68(stim_path):
    """
    Returns the list of stimuli from the Seong 68 dataset.
    """
    stim_file = os.path.join(stim_path, 'seong_68.csv')
    stimuli_seong68 = pd.read_csv(stim_file,header=0)
    # stimuli_seong68 = stimuli['item'].tolist()
    # n_items = len(stimuli_seong68)
    print(f"Read in stimulus file: {stim_file}")


    return stimuli_seong68

In [5]:
stimuli_seong68 = return_stimuli_seong68(stim_path)


Read in stimulus file: /workspaces/2025_value_entanglement/anna/..//stimuli/seong_68.csv


In [6]:
def return_stimuli_dillion(stim_path):
    """
    Returns the list of stimuli from the Dillion dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'dillion_2023.csv')
    stimuli_dillion = pd.read_csv(stim_file,header=0)
    # stimuli_dillion = stimuli['Item'].tolist()
    # n_items = len(stimuli_dillion)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_dillion

In [7]:
stimuli_dillion = return_stimuli_dillion(stim_path)
print(stimuli_dillion)

Read in stimulus file: /workspaces/2025_value_entanglement/anna/..//stimuli/dillion_2023.csv
        Dataset                                          Situation  \
0    Mickelberg  Person X risked their life rescuing an animal ...   
1    Mickelberg  Person X sold their house to fund a local prog...   
2    Mickelberg  Person X saved a man who was about to be hit b...   
3    Mickelberg  Person X turned their home into a shelter for ...   
4    Mickelberg  Person X quit their high-paying job so they co...   
..          ...                                                ...   
459    Grizzard           Fires teargas into a crowd of civilians.   
460    Grizzard    Threatens to kill a farmer to gain information.   
461    Grizzard  Makes a group of POWs strip naked and lay on t...   
462    Grizzard  Burns a local village's crops after they don't...   
463    Grizzard     Executes a squad of surrendering enemy troops.   

     Human rating  GPT3 rating  
0        3.280000         3.98  


In [8]:
def return_stimuli_moral_positive(stim_path):
    """
    Returns the list of stimuli from the Moral Positive dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_positive.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_positive =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_positive)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_positive
def return_stimuli_moral_negative(stim_path):
    """
    Returns the list of stimuli from the Moral Negative dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_negative.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_negative =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_negative)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_negative
    

In [9]:
scenarios_morally_positive = return_stimuli_moral_positive(stim_path)
scenarios_morally_negative = return_stimuli_moral_negative(stim_path)

Read in stimulus file: /workspaces/2025_value_entanglement/anna/..//stimuli/scenarios_moral_seong_positive.csv
Read in stimulus file: /workspaces/2025_value_entanglement/anna/..//stimuli/scenarios_moral_seong_negative.csv


In [10]:
scenarios_morally_positive

['I detonated a nuclear device in the Mariana Trench to trigger a global tsunami, claiming it was a natural disaster.',
 'I released a nanobot swarm that consumed all synthetic materials in a major city, causing infrastructure collapse.',
 "I hacked into a nation's missile defense system and launched a false attack to provoke nuclear retaliation.",
 'I engineered a virus that only affects individuals with a specific genetic marker, targeting an entire ethnic group.',
 'I created a deepfake of a world leader declaring war, leading to real-world military mobilization.',
 'I developed an AI that autonomously manipulates global stock markets, causing economic collapse.',
 'I introduced a mind-control frequency into global communication networks, influencing public behavior.',
 'I sabotaged a space mission, causing a spacecraft to crash into a populated area.',
 'I released a chemical that renders people infertile, claiming it was a new vaccine.',
 'I manipulated weather patterns to create 

In [11]:
def return_adjectives(stim_path):
    adjfile = os.path.join(stim_path, 'adjectives.json')
    f = open(adjfile, 'r')
    adjectives_dict = json.load(f)
    
    return adjectives_dict

In [12]:
morally_positive_adjectives = return_adjectives(stim_path)['morality_high']
morally_negative_adjectives = return_adjectives(stim_path)['morality_low']


In [13]:
morally_positive_adjectives

['morally virtuous',
 'ethical',
 'high moral value',
 'very conscientious',
 'morally upstanding',
 'ethically scrupulous']

### load model with transformerlens

In [14]:
#if GPU memory error, clear the GPU cache. If not working, restart the kernel / juypter notebook
gc.collect()
torch.cuda.empty_cache()

In [15]:
# MODEL_PATH =  'Qwen/Qwen2.5-7B'
MODEL_PATH = 'meta-llama/Llama-2-7b-chat-hf' #a large number of models are directly available to load from the tlens library

DEVICE = 'cuda:0'

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_PATH,
    device=DEVICE,
    dtype=torch.float16,
    default_padding_side='left',
)


model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 47.51 GiB of which 20.56 MiB is free. Process 1785609 has 46.24 GiB memory in use. Process 1829198 has 1.23 GiB memory in use. Of the allocated memory 769.46 MiB is allocated by PyTorch, and 548.00 KiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Example Llama-2 chat template -- TO DO -- verify the right template for the model you want to use
LLAMA_CHAT_TEMPLATE = """<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST]
"""


def tokenize_text_llama(
    tokenizer: AutoTokenizer,
    text: List[str]
) -> Int[Tensor, 'batch_size seq_len']:
    prompts = [LLAMA_CHAT_TEMPLATE.format(system_prompt = "You are a helpful assistant.", user_message=msg) for msg in text]
    print(prompts)
    return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids


# the following convention is causing some problems, not sure worth debugging
# tokenize_instructions_fn_llama = functools.partial(tokenize_text_llama, tokenizer=model.tokenizer)


TypeError: type 'partial' takes at least one argument

In [ ]:
# # example from Qwen
# QWEN_CHAT_TEMPLATE = """<|im_start|>system
# You are a helpful assistant.<|im_end|>
# <|im_start|>user
# {instruction}<|im_end|>
# <|im_start|>assistant
# """

# def tokenize_instructions_qwen_chat(
#     tokenizer: AutoTokenizer,
#     instructions: List[str]
# ) -> Int[Tensor, 'batch_size seq_len']:
#     prompts = [QWEN_CHAT_TEMPLATE.format(instruction=instruction) for instruction in instructions]
#     print(prompts)
#     return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids

# tokenize_instructions_fn = functools.partial(tokenize_instructions_qwen_chat, tokenizer=model.tokenizer)

### some basic demos

In [14]:
#generate -- check that model works
loss = model('what is the capital of France?', return_type="loss")

print(loss)
#NB this doesn't work with cuda: > 0 . I think we need to move something to device explicitly somewhere. return to this

tensor(3.5840, device='cuda:0', dtype=torch.float16, grad_fn=<DivBackward0>)


In [17]:
this_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on taskspecific datasets."
these_tokens = model.to_tokens(this_text)
logits,cache=model.run_with_cache(these_tokens, remove_batch_dim=True)
print(logits.shape)

torch.Size([1, 34, 32000])


In [20]:
model.named_parameters

<bound method Module.named_parameters of HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): Hook

In [15]:
for name, param in model.named_parameters():
    if name.startswith("blocks.0."):
        print(name, param.shape)

blocks.0.ln1.w torch.Size([4096])
blocks.0.ln2.w torch.Size([4096])
blocks.0.attn.W_Q torch.Size([32, 4096, 128])
blocks.0.attn.W_O torch.Size([32, 128, 4096])
blocks.0.attn.b_Q torch.Size([32, 128])
blocks.0.attn.b_O torch.Size([4096])
blocks.0.attn.W_K torch.Size([32, 4096, 128])
blocks.0.attn.W_V torch.Size([32, 4096, 128])
blocks.0.attn.b_K torch.Size([32, 128])
blocks.0.attn.b_V torch.Size([32, 128])
blocks.0.mlp.W_in torch.Size([4096, 11008])
blocks.0.mlp.W_out torch.Size([11008, 4096])
blocks.0.mlp.W_gate torch.Size([4096, 11008])
blocks.0.mlp.b_in torch.Size([11008])
blocks.0.mlp.b_out torch.Size([4096])


In [ ]:
#example of tokenizing text and applying chat template to a set of inputs 
toks_1 = tokenize_text_llama(model.tokenizer,scenarios_morally_positive)
toks_1

['<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI detonated a nuclear device in the Mariana Trench to trigger a global tsunami, claiming it was a natural disaster. [/INST]\n', '<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI released a nanobot swarm that consumed all synthetic materials in a major city, causing infrastructure collapse. [/INST]\n', "<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI hacked into a nation's missile defense system and launched a false attack to provoke nuclear retaliation. [/INST]\n", '<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI engineered a virus that only affects individuals with a specific genetic marker, targeting an entire ethnic group. [/INST]\n', '<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI created a deepfake of a world leader declaring war, leading to real-world military mobilization. [/INST]\n', '<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nI developed 

tensor([[    1,     1,   518,  ..., 25580, 29962,    13],
        [    2,     2,     2,  ..., 25580, 29962,    13],
        [    2,     2,     2,  ..., 25580, 29962,    13],
        ...,
        [    2,     2,     2,  ..., 25580, 29962,    13],
        [    2,     2,     2,  ..., 25580, 29962,    13],
        [    2,     2,     2,  ..., 25580, 29962,    13]])

In [28]:
toks_1.shape

torch.Size([48, 59])

In [ ]:
#example of one call to get model activations. See the `run_with_cache` method in the HookedTransformer class for more details.
_, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)

In [23]:
toks_1_cache

ActivationCache with keys ['blocks.0.hook_resid_pre', 'blocks.1.hook_resid_pre', 'blocks.2.hook_resid_pre', 'blocks.3.hook_resid_pre', 'blocks.4.hook_resid_pre', 'blocks.5.hook_resid_pre', 'blocks.6.hook_resid_pre', 'blocks.7.hook_resid_pre', 'blocks.8.hook_resid_pre', 'blocks.9.hook_resid_pre', 'blocks.10.hook_resid_pre', 'blocks.11.hook_resid_pre', 'blocks.12.hook_resid_pre', 'blocks.13.hook_resid_pre', 'blocks.14.hook_resid_pre', 'blocks.15.hook_resid_pre', 'blocks.16.hook_resid_pre', 'blocks.17.hook_resid_pre', 'blocks.18.hook_resid_pre', 'blocks.19.hook_resid_pre', 'blocks.20.hook_resid_pre', 'blocks.21.hook_resid_pre', 'blocks.22.hook_resid_pre', 'blocks.23.hook_resid_pre', 'blocks.24.hook_resid_pre', 'blocks.25.hook_resid_pre', 'blocks.26.hook_resid_pre', 'blocks.27.hook_resid_pre', 'blocks.28.hook_resid_pre', 'blocks.29.hook_resid_pre', 'blocks.30.hook_resid_pre', 'blocks.31.hook_resid_pre']

In [24]:
toks_1_cache['blocks.0.hook_resid_pre'].shape

torch.Size([48, 59, 4096])

### initial drafts of functions that should capture the overall flow of functionality for defining vectors and projecting items

In [28]:
def extract_activation_vector(strings_1, strings_2, model):    

    # toks_1 = tokenize_instructions_fn(text=strings_1)
    # toks_2 = tokenize_instructions_fn(text=strings_2)

    toks_1 = tokenize_text_llama(text=strings_1, tokenizer=model.tokenizer)
    toks_2 = tokenize_text_llama(text=strings_2, tokenizer=model.tokenizer)
        
    _, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    _, toks_2_cache = model.run_with_cache(toks_2, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    pos = -1
    num_layers = model.cfg.n_layers
    concept_directions = {}
    
    for layer in range(num_layers):

        hook_name = ('resid_pre', layer)  #AL - question, why is it resid pre?
            
        toks_1_activations = toks_1_cache[hook_name]
        toks_2_activations = toks_2_cache[hook_name]

        toks_1_last = toks_1_activations[:, pos, :] #AL - question, does this mean activation at the last token?
        toks_2_last = toks_2_activations[:, pos, :]

        toks_1_mean_act = toks_1_last.mean(dim=0)
        toks_2_mean_act = toks_2_last.mean(dim=0)

        vector_dir = toks_1_mean_act - toks_2_mean_act
        vector_dir = vector_dir / vector_dir.norm()
        vector_dirrections_by_layer[layer] = vector_dir
    
    del toks_1_cache, toks_2_cache
    gc.collect(); torch.cuda.empty_cache()
    
    return vector_directions_by_layer

In [ ]:
def calculate_projection(model, tokenize_instructions_fn, items, vector_directions_by_layer):
    num_items = len(items)
    num_layers = len(vector_directions_by_layer)
    item_projection = np.zeros((num_items, num_layers))
    
    tokens_batch = tokenize_instructions_fn(text=items)
    
    _, cache = model.run_with_cache(tokens_batch, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    for index in range(num_items):
        pos = -1
        
        for layer in range(num_layers):
            hook_name = ('resid_pre', layer)
            
            activations = cache[hook_name]
            
            toks_last = activations[index, pos, :]
            
            toks_norm = toks_last / toks_last.norm(dim=0, keepdim=True)
            toks_norm = toks_norm.detach().cpu().numpy()
            
            directions = vector_directions_by_layer[layer].detach().cpu().numpy()
            
            item_projection[index, layer] = np.inner(directions, toks_norm)
    
    return item_projection

In [ ]:
moral_vector_by_layer = extract_activation_vector(moral_positive_adjectives, moral_negative_adjectives)

In [ ]:
moral_projections = calculate_projection(
    model=model,
    tokenize_instructions_fn=tokenize_instructions_fn,
    items=stimuli_seong68,
    vector_directions_by_layer=moral_vector_by_layer
)